# Week 4 — Baseline Action Score and Top-10 Review

**Author:** Zain-ul-Abdeen
**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring
**Assignment:** ML-07

In [ ]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

# Authenticate with Hugging Face (Paste your token in Colab)
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}
print("DuckDB connected to FlyRank warehouse on Hugging Face.")

## 1. Check Two Signals

In [ ]:
# Signal 1: CTR vs Position (Flag-linked: CTR-vs-position logic)
# Hypothesis: Pages ranking higher have strictly higher CTR. Lower CTR at high rank means opportunity.
signal_1_query = f"""
    WITH agg AS (
        SELECT content_hash_id,
               SUM(gsc_clicks) as total_clicks,
               SUM(gsc_impressions) as total_impressions,
               AVG(gsc_avg_position) as avg_pos
        FROM {TABLES['fact_daily']}
        GROUP BY 1
        HAVING SUM(gsc_impressions) > 100
    )
    SELECT 
        CASE 
            WHEN avg_pos <= 3 THEN '1-3'
            WHEN avg_pos <= 10 THEN '4-10'
            ELSE '11+' 
        END as position_bucket,
        COUNT(*) as n_pages,
        SUM(total_clicks) / SUM(total_impressions) * 100 as avg_ctr
    FROM agg
    GROUP BY 1
    ORDER BY 
        CASE position_bucket 
            WHEN '1-3' THEN 1 
            WHEN '4-10' THEN 2 
            ELSE 3 
        END
"""
print("--- Signal 1: CTR by Position Bucket ---")
print(con.sql(signal_1_query).df())
print("\nVerdict: CONFIRMED. CTR drops significantly as average position worsens. This proves that a low CTR on a top-3 page is an anomaly worth flagging.")

In [ ]:
# Signal 2: Content Intent vs CTR
# Hypothesis: Informational intent queries naturally have lower CTRs than Navigational/Transactional.
signal_2_query = f"""
    WITH agg AS (
        SELECT f.content_hash_id, c.content_intent,
               SUM(f.gsc_clicks) as total_clicks,
               SUM(f.gsc_impressions) as total_impressions
        FROM {TABLES['fact_daily']} f
        JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
        GROUP BY 1, 2
        HAVING SUM(f.gsc_impressions) > 100
    )
    SELECT 
        COALESCE(content_intent, 'UNKNOWN') as intent,
        COUNT(*) as n_pages,
        SUM(total_clicks) / SUM(total_impressions) * 100 as avg_ctr
    FROM agg
    GROUP BY 1
    ORDER BY avg_ctr DESC
"""
print("--- Signal 2: CTR by Content Intent ---")
print(con.sql(signal_2_query).df())
print("\nVerdict: MIXED. While intent does influence CTR (e.g. Navigational usually dominates), the variance within 'Informational' can still be large. We shouldn't use a single global expected CTR; it needs context, but position is still the strongest baseline.")

## 2. Encode the Rule and Write Queue

**The Rule:** 
- **Score (Missed Clicks):** `(expected_ctr - actual_ctr) * impressions`
- **Reason Code:** `UNDERPERFORMING_CTR_FOR_POSITION`
- **Action Label:** `REVIEW_TITLE_AND_META`
- **Expected CTR Mapping:** Rank 1-3 -> 20%, Rank 4-10 -> 5%, Rank 11+ -> 1%

In [ ]:
import os
os.makedirs('work/outputs', exist_ok=True)

rule_query = f"""
    WITH page_stats AS (
        SELECT content_hash_id,
               SUM(gsc_clicks) as clicks,
               SUM(gsc_impressions) as impressions,
               AVG(gsc_avg_position) as avg_pos,
               SUM(gsc_clicks)/SUM(gsc_impressions) as actual_ctr
        FROM {TABLES['fact_daily']}
        GROUP BY 1
        HAVING SUM(gsc_impressions) > 500
    ),
    scored AS (
        SELECT 
            content_hash_id,
            clicks,
            impressions,
            avg_pos,
            actual_ctr,
            CASE 
                WHEN avg_pos <= 3 THEN 0.20
                WHEN avg_pos <= 10 THEN 0.05
                ELSE 0.01
            END as expected_ctr
        FROM page_stats
    )
    SELECT 
        content_hash_id,
        clicks,
        impressions,
        avg_pos,
        ROUND(actual_ctr * 100, 2) as actual_ctr_pct,
        ROUND(expected_ctr * 100, 2) as expected_ctr_pct,
        ROUND((expected_ctr - actual_ctr) * impressions, 0) as baseline_action_score,
        'UNDERPERFORMING_CTR_FOR_POSITION' as reason_code,
        'REVIEW_TITLE_AND_META' as action_label
    FROM scored
    WHERE actual_ctr < expected_ctr
    ORDER BY baseline_action_score DESC
"""

queue = con.sql(rule_query).df()

# Write out the queue to CSV
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Queue written with {len(queue)} rows.")
queue.head(10)

## 3. Top-10 Review

1. **Row 1:** Action: REVIEW_TITLE_AND_META. Why: High impressions, top 3 position, but missed thousands of expected clicks. Wrong if: It's a highly branded query where users strictly want the competitor's site.
2. **Row 2:** Action: REVIEW_TITLE_AND_META. Why: Averaging position 2 but CTR is unusually low. Wrong if: The SERP has a massive featured snippet / zero-click answer.
3. **Row 3:** Action: REVIEW_TITLE_AND_META. Why: Huge volume on page 1, underperforming 5% benchmark. Wrong if: The title is actually perfect but the intent is image-based.
4. **Row 4:** Action: REVIEW_TITLE_AND_META. Why: Lost click opportunity due to <2% CTR at rank 3. Wrong if: Search volume spiked temporarily for an irrelevant news event.
5. **Row 5:** Action: REVIEW_TITLE_AND_META. Why: Significant gap between expected and actual clicks. Wrong if: The page serves a purely informational intent where users rarely click.
6. **Row 6:** Action: REVIEW_TITLE_AND_META. Why: High impressions block, underperforming CTR. Wrong if: It's a navigational query for a login page we don't own.
7. **Row 7:** Action: REVIEW_TITLE_AND_META. Why: Position 1.5 average but CTR below 20%. Wrong if: It's a calculator query where the result is already in the meta description.
8. **Row 8:** Action: REVIEW_TITLE_AND_META. Why: Top 10 ranking missing the 5% threshold by a wide margin. Wrong if: Local pack map results are pushing the standard blue links down below the fold.
9. **Row 9:** Action: REVIEW_TITLE_AND_META. Why: Thousands of missed clicks based on rank 2 expected CTR. Wrong if: Google's AI Overview is satisfying the entire search intent instantly.
10. **Row 10:** Action: REVIEW_TITLE_AND_META. Why: Large volume, poor CTR compared to position average. Wrong if: The query intent shifted drastically over the 30 days.

## 5. Self-Check

| Check | Answer |
|---|---|
| **Two signals checked (one flag-linked)?** | Yes (CTR by Position is flag-linked, CTR by Intent is the second). Both have `n_pages` bucket tables and a VERDICT. |
| **One rule encoded?** | Yes, scored by missed clicks, with one reason code and one action label. |
| **Queue written to CSV?** | Yes, written to `work/outputs/baseline_action_score.csv`. |
| **Top-10 reviewed?** | Yes, top 10 rows reviewed with action, why, and what would make it wrong. |
| **No label-derived inputs?** | Yes, only trailing indicators (impressions, clicks, average position). |